# 1. Data Download

WBS 2.2 and 2.3. I download daily prices for the 16 ETFs, freeze them, run the quality checks and save the clean daily table.

In [ ]:
# Run first in every session: mount Drive, get the latest code, install libraries.
from google.colab import drive
drive.mount("/content/drive")

import os
import sys

REPO_URL = "https://github.com/guyen5/Capstone.git"
REPO_DIR = "/content/Capstone"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet {REPO_URL} {REPO_DIR}

%pip install --quiet -r {REPO_DIR}/requirements.txt

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
import importlib

import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf

from src import config, download, cleaning

# Reload so code pulled from GitHub takes effect without restarting.
for module in (config, download, cleaning):
    importlib.reload(module)

DIRS = config.data_dirs()

# One-time rename of files saved under the old naming scheme.
download.rename_old_files(DIRS)

print("Tickers:", len(config.ALL_TICKERS), config.ALL_TICKERS)
print("Window: ", config.START, "to", config.END, "(end exclusive)")
print("yfinance", yf.__version__)

## Download and freeze

Runs only when there is no frozen download yet. To replace it, set `NEW_DOWNLOAD = True` once.

In [ ]:
NEW_DOWNLOAD = False

if download.is_frozen(DIRS["raw"]) and not NEW_DOWNLOAD:
    print("Using the existing frozen download:", config.FILES["raw"])
else:
    prices_raw = download.download_prices()
    download.freeze(prices_raw, DIRS["raw"], overwrite=NEW_DOWNLOAD)

## Load the frozen file

In [ ]:
prices, info = download.load_frozen(DIRS["raw"])
print("Downloaded:", info["downloaded_utc"], "with yfinance", info["versions"]["yfinance"])
prices.head()

## Quality checks

A FAIL stops the notebook.

In [ ]:
report = cleaning.run_quality_checks(prices)
overall = cleaning.summarize(report)
report.to_csv(DIRS["reports"] / config.FILES["quality"], index=False)

if overall == "FAIL":
    raise RuntimeError("A quality check failed. Fix it before cleaning.")

## Clean and save

In [ ]:
clean, fill_log = cleaning.clean_prices(prices)

month_end_report = cleaning.check_month_ends(clean)
cleaning.summarize(month_end_report)
if (month_end_report["status"] == "FAIL").any():
    raise RuntimeError("A month-end price is missing.")

cleaning.save_clean(clean, fill_log, DIRS["processed"], DIRS["reports"])
print(f"{len(fill_log)} filled prices")

coverage = cleaning.coverage_table(clean)
coverage.to_csv(DIRS["reports"] / config.FILES["coverage"], index=False)
coverage

## Growth of $1

A vertical jump that is not a real market event would point to a bad price adjustment.

In [ ]:
adj = clean.pivot(index="date", columns="ticker", values="adj_close")
growth = adj / adj.iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (name, tickers) in zip(axes, [("ETF_A", config.ETF_A), ("ETF_B", config.ETF_B)]):
    growth[tickers].plot(ax=ax, logy=True, linewidth=1)
    ax.set_title(f"{name}: growth of $1 since 2009-01-02")
    ax.set_xlabel("")
    ax.legend(ncol=2, fontsize=8)
axes[0].set_ylabel("value of $1 (log scale)")
plt.tight_layout()
plt.show()

## Cross-check with Stooq (optional)

Runs only if `Stooq SPY.csv`, `Stooq AGG.csv` or `Stooq JPXN.csv` are in `data/raw`.

In [ ]:
rows = []
for ticker in ["SPY", "AGG", "JPXN"]:
    path = DIRS["raw"] / f"Stooq {ticker}.csv"
    if path.exists():
        rows.append(cleaning.cross_check(clean, path, ticker))
if rows:
    display(pd.DataFrame(rows))
else:
    print("No Stooq files, skipped.")

## Summary

In [ ]:
print(f"Raw file:      {config.FILES['raw']}")
print(f"MD5:           {info['md5']}")
print(f"Downloaded:    {info['downloaded_utc']}")
print(f"Rows:          {len(clean):,}")
print(f"Trading days:  {clean['date'].nunique():,}")
print(f"Month-ends:    {len(cleaning.month_end_dates(clean))}")
print(f"Checks:        {overall}, filled prices: {len(fill_log)}")